# Final Assignment: Analyzing Historical Stock/Revenue Data and Building a Dashboard

**Companies:** Tesla (TSLA) and GameStop (GME)

| Exercise | Task |
|---|---|
| 1 | Extract Tesla stock data with `yfinance` |
| 2 | Extract Tesla revenue data by web scraping |
| 3 | Extract GameStop stock data with `yfinance` |
| 4 | Extract GameStop revenue data by web scraping |
| 5 | Tesla stock and revenue dashboard |
| 6 | GameStop stock and revenue dashboard |

> Run **Kernel -> Restart & Run All** before submitting so every code cell shows its output.

In [ ]:
# Install dependencies (safe to re-run; remove the leading % if your environment already has them)
%pip install -q yfinance beautifulsoup4 lxml html5lib plotly requests pandas

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from io import StringIO

import pandas as pd
import requests
import yfinance as yf
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots

HEADERS = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"}

TESLA_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
GME_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"
print("Libraries imported.")

## Helper functions

`make_graph` draws the stock price and revenue in two stacked panels sharing a date axis.
`get_revenue_table` scrapes the revenue page and returns a clean `Date` / `Revenue` DataFrame.

In [ ]:
def make_graph(stock_data, revenue_data, stock):
    """Plot historical share price and historical revenue for `stock`."""
    stock_data_specific = stock_data[stock_data.Date <= "2021-06-14"]
    revenue_data_specific = revenue_data[revenue_data.Date <= "2021-04-30"]

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        subplot_titles=("Historical Share Price", "Historical Revenue"),
        vertical_spacing=0.3,
    )
    fig.add_trace(
        go.Scatter(x=pd.to_datetime(stock_data_specific.Date), y=stock_data_specific.Close.astype("float"),
                   name="Share Price"),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(x=pd.to_datetime(revenue_data_specific.Date), y=revenue_data_specific.Revenue.astype("float"),
                   name="Revenue"),
        row=2, col=1,
    )
    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Price ($US)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue ($US Millions)", row=2, col=1)
    fig.update_layout(showlegend=False, height=900, title=stock, xaxis_rangeslider_visible=True)
    fig.show()


def get_revenue_table(url):
    """Download `url`, locate the quarterly revenue table and return Date/Revenue columns."""
    html_data = requests.get(url, headers=HEADERS, timeout=30).text
    soup = BeautifulSoup(html_data, "html.parser")

    rows = []
    for table in soup.find_all("table"):
        header_text = table.get_text(" ", strip=True).lower()
        if "revenue" not in header_text:
            continue
        for tr in table.find("tbody").find_all("tr") if table.find("tbody") else table.find_all("tr"):
            cols = tr.find_all("td")
            if len(cols) >= 2:
                date = cols[0].get_text(strip=True)
                revenue = cols[1].get_text(strip=True)
                rows.append({"Date": date, "Revenue": revenue})
        if rows:          # first table that yields data is the quarterly revenue table
            break

    df = pd.DataFrame(rows)
    # Remove $ and thousands separators, drop empty values
    df["Revenue"] = df["Revenue"].str.replace(r"[\$,]", "", regex=True)
    df = df[df["Revenue"] != ""].dropna(subset=["Revenue"]).reset_index(drop=True)
    return df

## Exercise 1 - Extracting Tesla Stock Data Using yfinance
Use `yfinance.Ticker` to pull the full price history for Tesla (`TSLA`), reset the index so `Date` is a column, and display the first five rows.

In [ ]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)
tesla_data.head()

## Exercise 2 - Extracting Tesla Revenue Data Using Webscraping
Scrape the Tesla quarterly revenue table, clean the `$` and `,` characters and drop empty rows. Display the last five rows.

In [ ]:
tesla_revenue = get_revenue_table(TESLA_URL)
tesla_revenue.tail()

## Exercise 3 - Extracting GameStop Stock Data Using yfinance
Same process as Exercise 1, using the `GME` ticker.

In [ ]:
gme = yf.Ticker("GME")
gme_data = gme.history(period="max")
gme_data.reset_index(inplace=True)
gme_data.head()

## Exercise 4 - Extracting GameStop Revenue Data Using Webscraping
Scrape the GameStop quarterly revenue table and clean it the same way. Display the last five rows.

In [ ]:
gme_revenue = get_revenue_table(GME_URL)
gme_revenue.tail()

## Exercise 5 - Tesla Stock and Revenue Dashboard

In [ ]:
make_graph(tesla_data, tesla_revenue, "Tesla")

## Exercise 6 - GameStop Stock and Revenue Dashboard

In [ ]:
make_graph(gme_data, gme_revenue, "GameStop")